# Module 14 Instructor Colab Notebook: Graph Traversal and Optimization Algorithms

## A Mathematical Bridge to Artificial Intelligence

Module 13 described graph structure. This module asks algorithmic questions: How can we systematically visit every reachable vertex? How can we find a route with the fewest edges or smallest total cost? When do negative edge weights change the appropriate algorithm? When can a route use every edge exactly once, or visit every vertex exactly once?

We develop breadth-first search, depth-first search, Dijkstra's algorithm, Bellman–Ford, and Eulerian and Hamiltonian routes. Each algorithm is traced by hand before it is implemented.

**Recommended pacing:** three to four class meetings. Require students to record the changing queue, stack, distance labels, and parent pointers. Traversal order can depend on tie-breaking, so this notebook uses alphabetical neighbor order whenever a unique classroom trace is needed.


## Notebook map

| Section | Main mathematical ideas | Direct application |
|---|---|---|
| 1 | Graph representations, queues, stacks, relaxation | Algorithm foundations |
| 2 | Breadth-first search | Fewest-edge routes and graph layers |
| 3 | Depth-first search | Structural exploration and cycle detection |
| 4 | Dijkstra's algorithm | Nonnegative shortest paths |
| 5 | Bellman–Ford | Negative edges and negative-cycle detection |
| 6 | Eulerian and Hamiltonian routes | Street inspection and visit planning |
| Integrated lab | Robot routing network | Compare traversal and weighted optimization |


In [ ]:
# Core setup: compatible with Google Colab and standard Jupyter.
import math
import heapq
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

try:
    import ipywidgets as widgets
    from ipywidgets import interact, FloatSlider, IntSlider
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False

print("Setup complete.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)


In [ ]:
def reveal_markdown(button_text, solution_markdown):
    if not WIDGETS_AVAILABLE:
        display(Markdown("**Instructor solution**"))
        display(Markdown(solution_markdown))
        return

    button = widgets.Button(description=button_text, button_style="info")
    output = widgets.Output()
    state = {"open": False}

    def toggle(_):
        with output:
            output.clear_output()
            state["open"] = not state["open"]
            if state["open"]:
                display(Markdown(solution_markdown))
                button.description = "Hide instructor solution"
            else:
                button.description = button_text

    button.on_click(toggle)
    display(button, output)


def draw_graph(vertices, edges, positions, directed=False, weights=None,
               title="Graph", highlighted_edges=None, highlighted_vertices=None):
    highlighted_edges = set(highlighted_edges or [])
    highlighted_vertices = set(highlighted_vertices or [])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.axis("off")
    for index, edge in enumerate(edges):
        u, v = edge
        color = "crimson" if edge in highlighted_edges or (v, u) in highlighted_edges else "#475569"
        arrowstyle = "->" if directed else "-"
        ax.annotate("", xy=positions[v], xytext=positions[u],
                    arrowprops=dict(arrowstyle=arrowstyle, linewidth=2.5,
                                    color=color, shrinkA=18, shrinkB=18))
        if weights is not None:
            midpoint = ((positions[u][0]+positions[v][0])/2,
                        (positions[u][1]+positions[v][1])/2)
            ax.text(midpoint[0], midpoint[1]+0.05, str(weights[index]),
                    ha="center", va="center",
                    bbox=dict(facecolor="white", edgecolor="none", pad=1))
    for vertex in vertices:
        color = "#fecaca" if vertex in highlighted_vertices else "#dbeafe"
        ax.text(positions[vertex][0], positions[vertex][1], str(vertex),
                ha="center", va="center", fontsize=12,
                bbox=dict(boxstyle="circle,pad=0.45", facecolor=color, edgecolor="#1d4ed8"))
    ax.set_title(title)
    plt.show()


## Module learning objectives

By the end of the module, students should be able to:

1. explain how queues and stacks create different graph explorations;
2. trace and implement breadth-first search and depth-first search;
3. use BFS to find shortest paths in an unweighted graph;
4. carry out edge relaxation and Dijkstra's algorithm by hand;
5. explain why Dijkstra requires nonnegative edge weights;
6. carry out Bellman–Ford and detect a reachable negative cycle;
7. distinguish Eulerian from Hamiltonian questions; and
8. select an appropriate algorithm from graph and weight assumptions.


---

# Section 1: Algorithm Foundations

## 1.1 Input representations matter

An algorithm needs a data representation. For a graph with $$|V|$$ vertices and $$|E|$$ edges:

| Representation | Typical storage | Test whether $$u$$ and $$v$$ are adjacent | Visit all neighbors of $$u$$ |
|---|---:|---:|---:|
| Adjacency matrix | $$O(|V|^2)$$ | $$O(1)$$ | $$O(|V|)$$ |
| Adjacency list | $$O(|V|+|E|)$$ | Depends on list structure | $$O(\deg(u))$$ |

An adjacency list is usually preferred for a sparse graph, where the number of edges is much smaller than $$|V|^2.$$ An adjacency matrix can be convenient for dense graphs or matrix-based computations.

The notation $$O(g(n))$$ describes how a resource such as running time or memory grows at large input size, up to constant factors and lower-order terms. It is an asymptotic growth description, not an exact stopwatch time.


## 1.2 Three small data structures

- A **set** records which vertices have been discovered or visited.
- A **queue** is first-in, first-out: the earliest inserted item is removed first.
- A **stack** is last-in, first-out: the most recently inserted item is removed first.

Breadth-first search uses a queue and expands outward in layers. Depth-first search uses a stack, either explicitly or through recursive function calls, and follows one branch deeply before backtracking.


## 1.3 Parent pointers reconstruct a route

When an algorithm first reaches vertex $$v$$ from vertex $$u,$$ it can store

$$
\operatorname{parent}(v)=u.
$$

To reconstruct a route from a target to the source, repeatedly follow parent pointers backward and then reverse the sequence.

For example, if

$$
\operatorname{parent}(E)=D,
$$

$$
\operatorname{parent}(D)=B,
$$

and

$$
\operatorname{parent}(B)=A,
$$

then the backward chain is $$E,D,B,A,$$ and the forward path is

$$
A,B,D,E.
$$


In [ ]:
def adjacency_list(vertices, edges, directed=False):
    neighbors = {vertex: [] for vertex in vertices}
    for u, v in edges:
        neighbors[u].append(v)
        if not directed:
            neighbors[v].append(u)
    return {vertex: sorted(values) for vertex, values in neighbors.items()}


def reconstruct_path(parent, source, target):
    if target != source and target not in parent:
        return None
    path = [target]
    while path[-1] != source:
        path.append(parent[path[-1]])
    path.reverse()
    return path


parent_demo14 = {"B": "A", "D": "B", "E": "D"}
print(reconstruct_path(parent_demo14, "A", "E"))


---

# Section 2: Breadth-First Search

## 2.1 The central idea

**Breadth-first search (BFS)** explores every vertex at distance $$1$$ edge from the source, then every vertex at distance $$2,$$ and so on. The queue preserves this layer order.

For an unweighted graph, or a graph in which every edge has the same cost, BFS finds a path with the fewest edges from the source to every reachable vertex.

### BFS pseudocode

1. Mark the source discovered, give it distance $$0,$$ and enqueue it.
2. While the queue is not empty:
   1. dequeue a vertex $$u;$$
   2. examine every neighbor $$v$$ of $$u;$$
   3. if $$v$$ is undiscovered, mark it discovered, set its distance and parent, and enqueue it.

A vertex is marked when it is enqueued, not when it is later dequeued. This prevents the same vertex from entering the queue multiple times.


## 2.2 Detailed BFS trace

Use the undirected graph

$$
AB,\;AC,\;BD,\;BE,\;CF,\;EF
$$

and begin at $$A.$$ Process neighbors alphabetically.


In [ ]:
vertices_bfs14 = list("ABCDEF")
edges_bfs14 = [("A", "B"), ("A", "C"), ("B", "D"),
               ("B", "E"), ("C", "F"), ("E", "F")]
positions_bfs14 = {
    "A": (0, 1.2), "B": (-1, 0.3), "C": (1, 0.3),
    "D": (-1.7, -0.8), "E": (-0.5, -0.8), "F": (1.1, -0.8),
}
draw_graph(vertices_bfs14, edges_bfs14, positions_bfs14,
           title="Graph for BFS and DFS traces")


The complete BFS trace is:

| Step | Removed | Newly discovered | Queue after processing | Distances assigned |
|---:|---|---|---|---|
| 0 | — | $$A$$ | $$[A]$$ | $$d(A)=0$$ |
| 1 | $$A$$ | $$B,C$$ | $$[B,C]$$ | $$d(B)=d(C)=1$$ |
| 2 | $$B$$ | $$D,E$$ | $$[C,D,E]$$ | $$d(D)=d(E)=2$$ |
| 3 | $$C$$ | $$F$$ | $$[D,E,F]$$ | $$d(F)=2$$ |
| 4 | $$D$$ | none | $$[E,F]$$ | none |
| 5 | $$E$$ | none | $$[F]$$ | none |
| 6 | $$F$$ | none | $$[\;]$$ | none |

The BFS order is

$$
A,B,C,D,E,F.
$$

One shortest path from $$A$$ to $$F$$ is

$$
A,C,F,
$$

with length $$2.$$


In [ ]:
def bfs(vertices, edges, source):
    neighbors = adjacency_list(vertices, edges, directed=False)
    queue = deque([source])
    discovered = {source}
    parent = {}
    distance = {source: 0}
    order = []
    trace = []

    while queue:
        u = queue.popleft()
        order.append(u)
        new_vertices = []
        for v in neighbors[u]:
            if v not in discovered:
                discovered.add(v)
                parent[v] = u
                distance[v] = distance[u] + 1
                queue.append(v)
                new_vertices.append(v)
        trace.append((u, new_vertices, list(queue)))
    return order, parent, distance, trace


bfs_order14, bfs_parent14, bfs_distance14, bfs_trace14 = bfs(
    vertices_bfs14, edges_bfs14, "A"
)
print("Order:", bfs_order14)
print("Distance:", bfs_distance14)
print("Parent:", bfs_parent14)
print("Path A to F:", reconstruct_path(bfs_parent14, "A", "F"))


## 2.3 Why BFS gives fewest-edge distances

The queue processes vertices in nondecreasing distance from the source. All vertices at distance $$k$$ are processed before any vertex at distance $$k+1.$$

When BFS first discovers $$v$$ from $$u,$$ it assigns

$$
d(v)=d(u)+1.
$$

If a path with fewer edges existed, its previous vertex would lie in an earlier layer and would have discovered $$v$$ sooner. That contradicts the assumption that this is the first discovery. Hence the assigned distance is minimal.

With an adjacency list, each vertex is processed once and each undirected edge is examined from both endpoints, a constant number of times. Therefore the time complexity is

$$
\boxed{O(|V|+|E|).}
$$


## 2.4 BFS and AI systems

BFS is useful when edges have equal cost:

- finding the smallest number of relationship hops in a knowledge graph;
- finding degrees of separation in a social network;
- exploring a state space level by level;
- finding a shortest route on an unweighted grid.

For a very large graph, complete BFS can still be expensive because the number of reached vertices may grow rapidly with depth. Retrieval systems often impose depth limits, filter relation types, or use bidirectional and heuristic strategies.


## Classroom check 1

Using the displayed graph and alphabetical neighbors:

1. What are the BFS layers from $$A$$?
2. What is the fewest-edge distance from $$A$$ to $$E$$?
3. If the edge $$CF$$ had cost $$100,$$ would ordinary BFS minimize total cost?


In [ ]:
reveal_markdown("Open BFS solution", r"""
The layers are

$$
L_0=\{A\},
$$

$$
L_1=\{B,C\},
$$

$$
L_2=\{D,E,F\}.
$$

The fewest-edge distance from $$A$$ to $$E$$ is $$2,$$ using $$A,B,E.$$ Ordinary BFS would not necessarily minimize total cost when edge costs differ. It minimizes the number of edges, so a weighted shortest-path method is needed.
""")


---

# Section 3: Depth-First Search

## 3.1 The central idea

**Depth-first search (DFS)** follows one unexplored branch as far as possible and then backtracks to the most recent vertex with an unexplored neighbor. It uses a stack explicitly or uses the program's recursion stack.

### Recursive DFS pseudocode

1. Mark $$u$$ visited and process it.
2. For every neighbor $$v$$ of $$u$$:
   1. if $$v$$ is unvisited, set its parent to $$u$$ and call DFS on $$v.$$

Like BFS, DFS visits the entire connected component of its start vertex. Unlike BFS, it does not generally find fewest-edge paths.


## 3.2 Detailed DFS trace

Use the same graph, begin at $$A,$$ and choose alphabetical neighbors.

1. Visit $$A;$$ choose $$B.$$
2. At $$B,$$ skip already visited $$A;$$ choose $$D.$$
3. At $$D,$$ no unvisited neighbor remains; backtrack to $$B.$$
4. From $$B,$$ choose $$E.$$
5. From $$E,$$ choose $$F.$$
6. From $$F,$$ choose $$C.$$
7. At $$C,$$ all neighbors are visited; backtrack until the search ends.

The DFS preorder is

$$
A,B,D,E,F,C.
$$

The exact order can differ under another valid neighbor order. This is a tie-breaking difference, not a mathematical disagreement.


In [ ]:
def dfs(vertices, edges, source):
    neighbors = adjacency_list(vertices, edges, directed=False)
    visited = set()
    parent = {}
    preorder = []
    postorder = []

    def visit(u):
        visited.add(u)
        preorder.append(u)
        for v in neighbors[u]:
            if v not in visited:
                parent[v] = u
                visit(v)
        postorder.append(u)

    visit(source)
    return preorder, postorder, parent


dfs_preorder14, dfs_postorder14, dfs_parent14 = dfs(
    vertices_bfs14, edges_bfs14, "A"
)
print("DFS preorder:", dfs_preorder14)
print("DFS postorder:", dfs_postorder14)
print("DFS-tree path A to C:", reconstruct_path(dfs_parent14, "A", "C"))


The DFS-tree route from $$A$$ to $$C$$ is

$$
A,B,E,F,C,
$$

which has four edges. BFS found

$$
A,C,
$$

with one edge. DFS produced a valid search-tree path, but not the shortest path.


## 3.3 What DFS helps reveal

DFS supports many graph algorithms:

- finding connected components;
- detecting cycles;
- topological sorting of a directed acyclic graph;
- identifying bridges and articulation vertices;
- exploring puzzle or game state spaces with backtracking.

With adjacency lists, its time complexity is also

$$
\boxed{O(|V|+|E|).}
$$

Recursive code is compact, but a very deep graph can exceed a language's recursion limit. An explicit stack is then safer.


## 3.4 BFS and DFS comparison

| Feature | BFS | DFS |
|---|---|---|
| Main data structure | Queue | Stack or recursion |
| Exploration shape | Layer by layer | Deep branch, then backtrack |
| Fewest-edge path guarantee | Yes | No |
| Typical memory pressure | Large frontier | Deep active path |
| Common uses | Unweighted shortest paths, levels | Cycle structure, ordering, backtracking |

Both require a visited set on a general graph. Without it, a cycle can make an algorithm revisit vertices indefinitely.


## Classroom check 2

Explain why the statement “DFS is faster than BFS because it goes directly toward the target” is not generally valid.


In [ ]:
reveal_markdown("Open DFS solution", r"""
DFS has no knowledge of which branch leads to the target. It may explore a long irrelevant branch first. Both BFS and DFS have worst-case traversal time $$O(|V|+|E|)$$ with adjacency lists. Their practical behavior and memory use differ, and BFS has the fewest-edge guarantee while DFS does not.
""")


---

# Section 4: Dijkstra's Algorithm

## 4.1 Weighted shortest paths

For a path

$$
v_0,v_1,\ldots,v_k,
$$

its total weight is

$$
w(v_0,v_1)+w(v_1,v_2)+\cdots+w(v_{k-1},v_k).
$$

A **shortest path** minimizes total weight, not necessarily the number of edges. Dijkstra's algorithm finds single-source shortest paths when every edge weight is nonnegative:

$$
\boxed{w(e)\ge 0\text{ for every edge }e.}
$$

It maintains a tentative distance $$d(v)$$: the best source-to-$$v$$ cost found so far.


## 4.2 Relaxation

Suppose an algorithm knows a route to $$u$$ with cost $$d(u).$$ Edge $$u\to v$$ with weight $$w(u,v)$$ offers a candidate route to $$v$$ of cost

$$
d(u)+w(u,v).
$$

The **relaxation** operation is

$$
\boxed{d(v)\leftarrow\min\{d(v),d(u)+w(u,v)\}.}
$$

If the candidate is smaller, update both

$$
d(v)=d(u)+w(u,v)
$$

and

$$
\operatorname{parent}(v)=u.
$$

Relaxation is the shared mathematical heart of Dijkstra and Bellman–Ford. They differ in the order and number of relaxations.


## 4.3 Detailed Dijkstra example

Use the undirected weighted graph

$$
AB:4,\quad AC:1,\quad CB:2,\quad BD:1,
$$

$$
CD:5,\quad DE:3,\quad BE:7.
$$

Find shortest paths from $$A.$$


In [ ]:
vertices_dijkstra14 = list("ABCDE")
edges_dijkstra14 = [("A", "B"), ("A", "C"), ("C", "B"),
                    ("B", "D"), ("C", "D"), ("D", "E"), ("B", "E")]
weights_dijkstra14 = [4, 1, 2, 1, 5, 3, 7]
positions_dijkstra14 = {
    "A": (-2, 0), "C": (-0.8, 1), "B": (-0.5, -0.4),
    "D": (0.9, 0.5), "E": (2.0, -0.2),
}
draw_graph(vertices_dijkstra14, edges_dijkstra14, positions_dijkstra14,
           weights=weights_dijkstra14,
           title="Weighted graph for Dijkstra's algorithm")


Initialize

$$
d(A)=0
$$

and

$$
d(B)=d(C)=d(D)=d(E)=\infty.
$$

At each step, **settle** the unsettled vertex with smallest tentative distance, then relax its outgoing edges.

| Settled vertex | Important relaxations | Distances after step $$[A,B,C,D,E]$$ |
|---|---|---|
| $$A$$ | $$B: \infty\to4,$$ $$C:\infty\to1$$ | $$[0,4,1,\infty,\infty]$$ |
| $$C$$ | $$B:4\to1+2=3,$$ $$D:\infty\to1+5=6$$ | $$[0,3,1,6,\infty]$$ |
| $$B$$ | $$D:6\to3+1=4,$$ $$E:\infty\to3+7=10$$ | $$[0,3,1,4,10]$$ |
| $$D$$ | $$E:10\to4+3=7$$ | $$[0,3,1,4,7]$$ |
| $$E$$ | no improvement | $$[0,3,1,4,7]$$ |

Therefore,

$$
d(A)=0,\quad d(C)=1,\quad d(B)=3,
$$

$$
d(D)=4,\quad d(E)=7.
$$

The shortest route to $$E$$ is

$$
A,C,B,D,E
$$

with total weight

$$
1+2+1+3=7.
$$


In [ ]:
def weighted_adjacency(vertices, edges, weights, directed=False):
    neighbors = {vertex: [] for vertex in vertices}
    for (u, v), weight in zip(edges, weights):
        neighbors[u].append((v, weight))
        if not directed:
            neighbors[v].append((u, weight))
    for vertex in neighbors:
        neighbors[vertex].sort(key=lambda pair: str(pair[0]))
    return neighbors


def dijkstra(vertices, edges, weights, source, directed=False):
    if any(weight < 0 for weight in weights):
        raise ValueError("Dijkstra requires nonnegative edge weights.")
    neighbors = weighted_adjacency(vertices, edges, weights, directed)
    distance = {vertex: math.inf for vertex in vertices}
    parent = {}
    distance[source] = 0
    priority_queue = [(0, source)]
    settled = set()
    trace = []

    while priority_queue:
        current_distance, u = heapq.heappop(priority_queue)
        if u in settled:
            continue
        settled.add(u)
        changes = []
        for v, weight in neighbors[u]:
            candidate = current_distance + weight
            if candidate < distance[v]:
                old = distance[v]
                distance[v] = candidate
                parent[v] = u
                heapq.heappush(priority_queue, (candidate, v))
                changes.append((v, old, candidate))
        trace.append((u, current_distance, changes, dict(distance)))
    return distance, parent, trace


dijkstra_distance14, dijkstra_parent14, dijkstra_trace14 = dijkstra(
    vertices_dijkstra14, edges_dijkstra14, weights_dijkstra14, "A"
)
print("Distances:", dijkstra_distance14)
print("Path A to E:", reconstruct_path(dijkstra_parent14, "A", "E"))


## 4.4 Why nonnegative weights make the greedy choice safe

When Dijkstra settles the smallest tentative distance $$d(u),$$ every unsettled route must first reach some unsettled frontier vertex. Adding another nonnegative edge cannot reduce that route's cost below what has already been accumulated.

Therefore no later detour can produce a route to $$u$$ cheaper than the current smallest label. The label is final.

With a binary-heap priority queue and adjacency lists, a common complexity bound is

$$
O((|V|+|E|)\log |V|),
$$

often simplified to

$$
O(|E|\log |V|)
$$

for a connected sparse graph.


## 4.5 Why a negative edge breaks the argument

Suppose

$$
A\to B=2,\qquad A\to C=5,\qquad C\to B=-10.
$$

Dijkstra would initially see $$d(B)=2$$ and $$d(C)=5$$ and may settle $$B$$ first. But the later route

$$
A\to C\to B
$$

has cost

$$
5+(-10)=-5,
$$

which is smaller than $$2.$$ A negative edge can make a later route undo previously accumulated cost. This destroys the greedy guarantee.


## Classroom check 3

In the weighted graph above:

1. Why does BFS prefer $$A,B,E$$ to $$A,C,B,D,E$$ when measured only in edges?
2. What are the two route costs?
3. Which algorithm gives the minimum total weight?


In [ ]:
reveal_markdown("Open Dijkstra solution", r"""
BFS minimizes edge count. Route $$A,B,E$$ has two edges and weight

$$
4+7=11.
$$

Route $$A,C,B,D,E$$ has four edges and weight

$$
1+2+1+3=7.
$$

Dijkstra's algorithm accounts for the nonnegative weights and correctly selects the second route as the minimum-weight path.
""")


---

# Section 5: Bellman–Ford

## 5.1 Repeated relaxation instead of a greedy finalization

The **Bellman–Ford algorithm** finds single-source shortest paths in a directed graph even when some edges have negative weights, provided no negative-weight cycle is reachable from the source and can affect the requested destination.

It performs the following steps:

1. Set the source distance to $$0$$ and all others to $$\infty.$$
2. Repeat $$|V|-1$$ times: relax every directed edge.
3. Make one additional pass. If any distance can still decrease, report a reachable negative cycle.

The edge order can affect intermediate passes, but not the correct final distances after sufficient passes.


## 5.2 Why $$|V|-1$$ passes are sufficient

If no useful negative cycle exists, a shortest path can be chosen to be simple: it repeats no vertex. A simple path in a graph with $$|V|$$ vertices uses at most

$$
|V|-1
$$

edges.

After one full pass, relaxation can correctly propagate information along at least one more edge of a shortest path; after $$|V|-1$$ passes, every simple shortest path has had enough opportunities. Some implementations stop early if a complete pass makes no change.


## 5.3 Detailed Bellman–Ford example

Use directed edges in this fixed order:

$$
A\to B:4,
$$

$$
A\to C:5,
$$

$$
B\to C:-2,
$$

$$
C\to D:3,
$$

$$
B\to D:4.
$$

Initialize

$$
[d(A),d(B),d(C),d(D)]=[0,\infty,\infty,\infty].
$$

During pass 1:

$$
A\to B:\quad d(B)=\min(\infty,0+4)=4,
$$

$$
A\to C:\quad d(C)=\min(\infty,0+5)=5,
$$

$$
B\to C:\quad d(C)=\min(5,4-2)=2,
$$

$$
C\to D:\quad d(D)=\min(\infty,2+3)=5,
$$

$$
B\to D:\quad d(D)=\min(5,4+4)=5.
$$

Thus the first pass already produces

$$
[0,4,2,5].
$$

The next pass makes no improvement, so the algorithm may stop early.


In [ ]:
def bellman_ford(vertices, edges, weights, source):
    distance = {vertex: math.inf for vertex in vertices}
    parent = {}
    distance[source] = 0
    trace = []

    for pass_number in range(len(vertices) - 1):
        changes = []
        for (u, v), weight in zip(edges, weights):
            if distance[u] != math.inf and distance[u] + weight < distance[v]:
                old = distance[v]
                distance[v] = distance[u] + weight
                parent[v] = u
                changes.append((u, v, old, distance[v]))
        trace.append((pass_number + 1, changes, dict(distance)))
        if not changes:
            break

    negative_cycle = any(
        distance[u] != math.inf and distance[u] + weight < distance[v]
        for (u, v), weight in zip(edges, weights)
    )
    return distance, parent, negative_cycle, trace


vertices_bf14 = list("ABCD")
edges_bf14 = [("A", "B"), ("A", "C"), ("B", "C"), ("C", "D"), ("B", "D")]
weights_bf14 = [4, 5, -2, 3, 4]

bf_distance14, bf_parent14, bf_negative_cycle14, bf_trace14 = bellman_ford(
    vertices_bf14, edges_bf14, weights_bf14, "A"
)
print("Distances:", bf_distance14)
print("Path A to D:", reconstruct_path(bf_parent14, "A", "D"))
print("Reachable negative cycle?", bf_negative_cycle14)
print("Pass trace:", bf_trace14)


## 5.4 Negative cycles

A **negative-weight cycle** is a directed cycle whose total edge weight is negative. If it is reachable from the source and lies on a route relevant to the destination, there is no finite shortest route: going around the cycle repeatedly makes the total cost arbitrarily small.

For example, if

$$
B\to C=-2
$$

and

$$
C\to B=1,
$$

then the cycle has total weight

$$
-2+1=-1.
$$

After $$k$$ trips around it, the cost decreases by $$k.$$ Bellman–Ford's extra relaxation pass detects that some label can still decrease.


In [ ]:
negative_edges14 = [("A", "B"), ("B", "C"), ("C", "B")]
negative_weights14 = [1, -2, 1]
negative_result14 = bellman_ford(
    ["A", "B", "C"], negative_edges14, negative_weights14, "A"
)
print("Reachable negative cycle detected?", negative_result14[2])


## 5.5 Dijkstra versus Bellman–Ford

| Question | Dijkstra | Bellman–Ford |
|---|---|---|
| Allows negative edges? | No | Yes |
| Detects reachable negative cycles? | No | Yes |
| Main strategy | Greedily settle smallest label | Repeatedly relax all edges |
| Typical time with common structures | $$O((|V|+|E|)\log|V|)$$ | $$O(|V||E|)$$ |
| Preferred when | All weights are nonnegative | Negative weights may occur |

If every edge has equal weight, BFS is simpler and faster. Algorithm selection begins with the graph's assumptions.


## Classroom check 4

Choose BFS, Dijkstra, or Bellman–Ford.

1. Every edge represents one social-network hop.
2. Road travel times are all nonnegative and unequal.
3. A directed financial model contains positive and negative adjustments.
4. Explain why “use Bellman–Ford for everything” is mathematically possible in some cases but computationally unattractive.


In [ ]:
reveal_markdown("Open Bellman\u2013Ford solution", r"""
1. Use BFS because every edge has equal cost.
2. Use Dijkstra because weights differ but are nonnegative.
3. Use Bellman–Ford because negative weights may occur and negative-cycle detection matters.
4. Bellman–Ford can solve nonnegative instances, but its $$O(|V||E|)$$ running time is generally worse than the common heap-based Dijkstra bound, and BFS is simpler for equal weights.
""")


---

# Section 6: Eulerian and Hamiltonian Routes

## 6.1 Similar names, different objects

- An **Eulerian trail** uses every edge exactly once.
- An **Eulerian circuit** is an Eulerian trail that returns to its starting vertex.
- A **Hamiltonian path** visits every vertex exactly once.
- A **Hamiltonian cycle** is a Hamiltonian path that returns to its starting vertex.

Eulerian questions focus on edges. Hamiltonian questions focus on vertices.

This creates different applications. A street inspector who must traverse every street is solving an edge-coverage problem. A delivery planner who wants to visit each destination once is solving a vertex-visit problem.


## 6.2 Euler's degree criteria for undirected graphs

Ignoring isolated vertices, assume the graph is connected.

1. An Eulerian circuit exists exactly when every vertex has even degree.
2. An Eulerian trail with different start and end exists exactly when exactly two vertices have odd degree.
3. If more than two vertices have odd degree, no Eulerian trail exists.

### Why odd degrees determine the endpoints

Every time a trail enters an internal vertex along one unused edge, it must leave along another unused edge. Those edges occur in pairs. The start can have one unpaired departure, and the end can have one unpaired arrival. Therefore only the two endpoints may have odd degree.


## 6.3 Detailed Eulerian example

Consider edges

$$
AB,\;BC,\;CD,\;DA,\;AC.
$$

The degrees are

$$
\deg(A)=3,\quad \deg(B)=2,
$$

$$
\deg(C)=3,\quad \deg(D)=2.
$$

Exactly $$A$$ and $$C$$ have odd degree. Therefore an Eulerian trail exists and must begin at one of these vertices and end at the other.

One such trail is

$$
A,B,C,D,A,C.
$$

It uses all five edges exactly once. It is not an Eulerian circuit because its endpoints differ.


## 6.4 Directed Eulerian criteria

For a directed Eulerian circuit, every non-isolated vertex must belong to the appropriate connected directed structure and satisfy

$$
\deg^+(v)=\deg^-(v)
$$

for every vertex $$v.$$

For an open directed Eulerian trail, the start has one extra outgoing edge,

$$
\deg^+(s)=\deg^-(s)+1,
$$

the end has one extra incoming edge,

$$
\deg^-(t)=\deg^+(t)+1,
$$

and every other vertex has equal in-degree and out-degree. Connectivity conditions still matter; balanced degrees alone do not join separated pieces.


## 6.5 Hamiltonian problems are harder

There is no simple degree-parity test that completely characterizes Hamiltonian paths and cycles in arbitrary graphs. Necessary or sufficient theorems exist, but the general decision problem is computationally difficult.

Determining whether an arbitrary graph has a Hamiltonian cycle is **NP-complete**. Informally, a proposed cycle can be checked efficiently, but no efficient algorithm is known that always finds or rejects one for every graph.

Do not confuse a Hamiltonian cycle with the traveling salesperson problem. The Hamiltonian question asks whether a visit-once cycle exists. The traveling salesperson problem assigns weights and asks for the minimum-weight tour.


## 6.6 A graph can have one kind but not the other

The star graph with center $$C$$ and leaves $$A,B,D,E$$ has degrees

$$
\deg(C)=4
$$

and

$$
\deg(A)=\deg(B)=\deg(D)=\deg(E)=1.
$$

It has four odd-degree vertices, so it has no Eulerian trail. It also has no Hamiltonian path through all five vertices: after visiting one leaf and the center, a route can visit only one additional leaf without revisiting the center.

By contrast, a simple cycle $$C_n$$ is both Eulerian and Hamiltonian: it uses every edge and visits every vertex exactly once before returning.


## Classroom check 5

For a connected undirected graph with degrees

$$
4,2,2,2,
$$

what Eulerian conclusion follows? Does the degree list alone settle whether a Hamiltonian cycle exists?


In [ ]:
reveal_markdown("Open route-classification solution", r"""
All degrees are even, so the connected graph has an Eulerian circuit. The degree list alone does not settle Hamiltonicity. Hamiltonian structure depends on how vertices are connected, and general degree sequences do not provide a complete test.
""")


---

# Integrated Lab: Robot Routing Through a Building

A robot moves among locations

$$
S,A,B,C,D,G,
$$

where $$S$$ is the start and $$G$$ is the goal. Edge weights are travel times in seconds:

$$
SA:2,\quad SB:5,\quad AB:1,\quad AC:4,
$$

$$
BC:1,\quad BD:4,\quad CD:1,\quad DG:3,\quad CG:6.
$$

All corridors are two-way, and all weights are nonnegative.


In [ ]:
robot_vertices14 = ["S", "A", "B", "C", "D", "G"]
robot_edges14 = [("S", "A"), ("S", "B"), ("A", "B"), ("A", "C"),
                 ("B", "C"), ("B", "D"), ("C", "D"), ("D", "G"), ("C", "G")]
robot_weights14 = [2, 5, 1, 4, 1, 4, 1, 3, 6]
robot_positions14 = {
    "S": (-2.2, 0), "A": (-1.2, 0.9), "B": (-1.0, -0.5),
    "C": (0.3, 0.8), "D": (0.5, -0.6), "G": (2.0, 0),
}

draw_graph(robot_vertices14, robot_edges14, robot_positions14,
           weights=robot_weights14, title="Robot travel-time network")


## Integrated-lab tasks

1. Ignore weights and run BFS from $$S.$$ Find a fewest-edge path to $$G.$$
2. Run Dijkstra from $$S.$$ Record every final distance and reconstruct a minimum-time path to $$G.$$
3. Compare the edge count and travel time of the BFS path with the Dijkstra path.
4. Explain why Bellman–Ford is valid but unnecessary here.
5. Determine whether the graph has an Eulerian trail or circuit.
6. Identify one Hamiltonian path, if one exists.


In [ ]:
robot_bfs_order14, robot_bfs_parent14, robot_bfs_distance14, _ = bfs(
    robot_vertices14, robot_edges14, "S"
)
robot_dijkstra_distance14, robot_dijkstra_parent14, _ = dijkstra(
    robot_vertices14, robot_edges14, robot_weights14, "S"
)
robot_bfs_path14 = reconstruct_path(robot_bfs_parent14, "S", "G")
robot_dijkstra_path14 = reconstruct_path(robot_dijkstra_parent14, "S", "G")

def path_weight(path, edges, weights):
    edge_weight = {}
    for (u, v), weight in zip(edges, weights):
        edge_weight[(u, v)] = weight
        edge_weight[(v, u)] = weight
    return sum(edge_weight[(u, v)] for u, v in zip(path[:-1], path[1:]))


print("BFS order:", robot_bfs_order14)
print("Fewest-edge path:", robot_bfs_path14)
print("Its travel time:", path_weight(robot_bfs_path14, robot_edges14, robot_weights14))
print("Dijkstra distances:", robot_dijkstra_distance14)
print("Minimum-time path:", robot_dijkstra_path14)
print("Minimum travel time:", robot_dijkstra_distance14["G"])


In [ ]:
reveal_markdown("Open integrated-lab solution", r"""
With alphabetical neighbors, BFS discovers the three-edge route

$$
S,A,C,G.
$$

It contains four listed vertices but three traversed edges. Its travel time is

$$
2+4+6=12.
$$

Dijkstra improves the distances in stages:

$$
d(A)=2,
$$

$$
d(B)=2+1=3,
$$

$$
d(C)=3+1=4,
$$

$$
d(D)=4+1=5,
$$

$$
d(G)=5+3=8.
$$

Thus the minimum-time path is

$$
S,A,B,C,D,G
$$

with total time

$$
2+1+1+1+3=8.
$$

Bellman–Ford would also find the distances, but Dijkstra is preferred because all weights are nonnegative.

The degrees in the listed edge order are

$$
\deg(S)=2,\quad \deg(A)=3,\quad \deg(B)=4,
$$

$$
\deg(C)=4,\quad \deg(D)=3,\quad \deg(G)=2.
$$

Exactly $$A$$ and $$D$$ are odd, so an Eulerian trail exists from one to the other, but no Eulerian circuit exists. One Hamiltonian path is

$$
S,A,B,C,D,G.
$$
""")


> **Instructor emphasis:** The BFS route has four listed vertices but three traversed edges. Counting edges rather than vertices is part of the assessment.


---

# Algorithm Selection Guide

| Situation | Recommended starting algorithm | Reason |
|---|---|---|
| Visit all reachable vertices by layers | BFS | Queue produces increasing hop distance |
| Explore deeply, backtrack, or analyze structure | DFS | Stack/recursion follows branches |
| Shortest path with equal edge costs | BFS | Hop count equals total cost up to a constant |
| Shortest path with unequal nonnegative weights | Dijkstra | Greedy settlement is valid |
| Shortest path with possible negative edges | Bellman–Ford | Repeated relaxation remains valid |
| Use every edge once | Eulerian analysis | Degree balance characterizes existence in common cases |
| Visit every vertex once | Hamiltonian analysis | A distinct and generally harder problem |

Real robot motion planning may also require geometry, collision checking, uncertainty, dynamic obstacles, and heuristic algorithms such as A*. The graph algorithms here are foundational components, not a complete autonomous-driving system.


# Common Mistakes and Corrections

1. **Mistake:** Marking a BFS vertex only when it is dequeued.
   **Correction:** Mark it when enqueued to prevent duplicate queue entries.
2. **Mistake:** Assuming DFS returns a shortest path.
   **Correction:** DFS returns a search-tree path; BFS gives fewest edges in an unweighted graph.
3. **Mistake:** Counting vertices instead of edges when measuring path length.
   **Correction:** A path with $$k+1$$ listed vertices has $$k$$ edges.
4. **Mistake:** Applying Dijkstra when a negative edge is present.
   **Correction:** The greedy finalization proof requires nonnegative weights.
5. **Mistake:** Treating a reachable negative cycle as a very short finite route.
   **Correction:** Repeating it makes cost unbounded below, so no finite minimum exists.
6. **Mistake:** Confusing Eulerian with Hamiltonian.
   **Correction:** Eulerian uses every edge; Hamiltonian visits every vertex.
7. **Mistake:** Ignoring disconnected pieces in Eulerian criteria.
   **Correction:** Degree conditions must be paired with the appropriate connectivity condition.
8. **Mistake:** Treating tie-dependent traversal order as unique.
   **Correction:** State a neighbor-order rule; structural results can remain valid under different orders.


---

# Module 14 Summary

1. BFS uses a queue, explores layers, and finds fewest-edge paths.
2. DFS uses a stack or recursion, explores deeply, and supports structural algorithms.
3. Parent pointers reconstruct algorithmic routes.
4. Relaxation tests whether a known route can improve a distance label.
5. Dijkstra is efficient for nonnegative weights because settled labels are final.
6. Bellman–Ford handles negative edges and detects reachable negative cycles.
7. Eulerian routes cover edges, while Hamiltonian routes cover vertices.
8. Correct algorithm selection begins with direction, connectivity, and edge-weight assumptions.

## Exit ticket

1. Why does BFS need a queue?
2. Give one task for which DFS is more natural than BFS.
3. What exactly happens during relaxation?
4. Which assumption makes Dijkstra's greedy step safe?
5. Why does Bellman–Ford make an extra pass?
6. What is the fastest verbal test for separating Eulerian and Hamiltonian questions?
